In [1]:
# ===== РАЗДЕЛ 1.6 — инструменты =====
# Три способа учёта погрешностей:
#   I  — метод границ (интервальная арифметика)
#   E  — пооперационный учёт погрешностей (значение ± Δ)
#   diff_err — итоговая оценка через полный дифференциал

import math
import itertools


class I:
    """Метод границ: число задано отрезком [lo; hi]."""

    def __init__(s, lo, hi=None):
        s.lo, s.hi = (lo, lo) if hi is None else (min(lo, hi), max(lo, hi))

    def __repr__(s):
        return f"[{s.lo:.10g}; {s.hi:.10g}]"

    def __add__(s, o):
        o = W(o); return I(s.lo + o.lo, s.hi + o.hi)

    def __radd__(s, o):
        return W(o) + s

    def __sub__(s, o):
        o = W(o); return I(s.lo - o.hi, s.hi - o.lo)

    def __rsub__(s, o):
        return W(o) - s

    def __mul__(s, o):
        o = W(o)
        p = [s.lo * o.lo, s.lo * o.hi, s.hi * o.lo, s.hi * o.hi]
        return I(min(p), max(p))

    def __rmul__(s, o):
        return W(o) * s

    def __truediv__(s, o):
        o = W(o)
        if o.lo <= 0 <= o.hi:
            raise ZeroDivisionError("ноль внутри знаменателя")
        p = [s.lo / o.lo, s.lo / o.hi, s.hi / o.lo, s.hi / o.hi]
        return I(min(p), max(p))

    def __rtruediv__(s, o):
        return W(o) / s

    def mid(s):
        return (s.lo + s.hi) / 2

    def d(s):
        return (s.hi - s.lo) / 2


def W(x):
    return x if isinstance(x, I) else I(x)


def mono(f, x):
    return I(f(x.lo), f(x.hi))


def isqrt(x):  return mono(math.sqrt, x)
def icbrt(x):  return mono(lambda t: math.copysign(abs(t) ** (1 / 3), t), x)
def iexp(x):   return mono(math.exp, x)
def iln(x):    return mono(math.log, x)
def ilg(x):    return mono(math.log10, x)


def ipow(a, b):
    a, b = W(a), W(b)
    p = [a.lo ** b.lo, a.lo ** b.hi, a.hi ** b.lo, a.hi ** b.hi]
    return I(min(p), max(p))


def _ext(f, x, crit):
    """Экстремумы тригонометрии внутри отрезка."""
    v = [f(x.lo), f(x.hi)]
    k = math.floor(x.lo / math.pi) - 1
    while k * math.pi <= x.hi + math.pi:
        for c in crit:
            t = k * math.pi + c
            if x.lo <= t <= x.hi:
                v.append(f(t))
        k += 1
    return I(min(v), max(v))


def isin(x): return _ext(math.sin, x, [math.pi / 2])
def icos(x): return _ext(math.cos, x, [0.0])


class E:
    """Пооперационный учёт: значение и накопленная абсолютная погрешность."""

    def __init__(s, v, d=0.0):
        s.v, s.d = v, d

    def __repr__(s):
        return f"{s.v:.8g} +- {s.d:.3g}"

    def rel(s):
        return s.d / abs(s.v) if s.v else 0.0

    def __add__(s, o):
        o = V(o); return E(s.v + o.v, s.d + o.d)          # Δ складываются

    def __radd__(s, o):
        return V(o) + s

    def __sub__(s, o):
        o = V(o); return E(s.v - o.v, s.d + o.d)          # Δ складываются и здесь

    def __rsub__(s, o):
        return V(o) - s

    def __mul__(s, o):
        o = V(o); return E(s.v * o.v, abs(s.v * o.v) * (s.rel() + o.rel()))   # δ складываются

    def __rmul__(s, o):
        return V(o) * s

    def __truediv__(s, o):
        o = V(o); return E(s.v / o.v, abs(s.v / o.v) * (s.rel() + o.rel()))

    def __rtruediv__(s, o):
        return V(o) / s


def V(x):
    return x if isinstance(x, E) else E(x)


def epow(x, p):
    if isinstance(p, E):
        v = x.v ** p.v
        return E(v, abs(v) * (abs(p.v) * x.rel() + abs(math.log(x.v)) * p.d))
    v = x.v ** p
    return E(v, abs(v) * abs(p) * x.rel())


def esqrt(x): v = math.sqrt(x.v);        return E(v, v * x.rel() / 2)
def ecbrt(x): v = abs(x.v) ** (1 / 3);   return E(math.copysign(v, x.v), v * x.rel() / 3)
def eexp(x):  v = math.exp(x.v);         return E(v, v * x.d)
def eln(x):   return E(math.log(x.v),   x.d / abs(x.v))
def elg(x):   return E(math.log10(x.v), x.d / (abs(x.v) * math.log(10)))
def esin(x):  return E(math.sin(x.v), abs(math.cos(x.v)) * x.d)
def ecos(x):  return E(math.cos(x.v), abs(math.sin(x.v)) * x.d)


def num_i(v, dec):
    """Интервал для числа, у которого верны все цифры до разряда 10^dec."""
    return I(v - 0.5 * 10 ** dec, v + 0.5 * 10 ** dec)


def diff_err(f, args, deltas, h=1e-7):
    """Итоговая оценка: Δy = Σ |∂f/∂xi| · Δxi."""
    y = f(*args)
    tot = 0.0
    for i in range(len(args)):
        p = list(args); p[i] += h; d1 = f(*p)
        p = list(args); p[i] -= h; d2 = f(*p)
        tot += abs((d1 - d2) / (2 * h)) * deltas[i]
    return y, tot


def verified(y, dy):
    """Разряд, округлённое значение и число верных цифр в строгом смысле."""
    m = -12
    while 0.5 * 10 ** m < dy:
        m += 1
    r = round(y, -m) if m <= 0 else round(y / 10 ** m) * 10 ** m
    s = f"{abs(r):.{max(0, -m)}f}".replace(".", "").lstrip("0")
    return m, r, len(s) if s else 1


print("Инструменты готовы: I (границы), E (пооперационно), diff_err (итоговая оценка).")


Инструменты готовы: I (границы), E (пооперационно), diff_err (итоговая оценка).


In [2]:
# ===== УПРАЖНЕНИЕ 1.6.1 =====
# Вычислить значения выражений двумя способами:
#   1) по правилам подсчёта цифр (метод границ — пооперационно);
#   2) с итоговой оценкой окончательного результата.
# У всех числовых данных все цифры верные.

print("=" * 78)
print("УПРАЖНЕНИЕ 1.6.1")
print("=" * 78)


def task_161(name, box, f, args, deltas):
    y, dy = diff_err(f, args, deltas)
    m1, r1, n1 = verified(box.mid(), box.d())
    m2, r2, n2 = verified(y, dy)
    print(f"\n{name}")
    print(f"   1) метод границ:     {box}")
    print(f"      середина {box.mid():.8g},  Δ = {box.d():.3g}   ->  {r1}   ({n1} верных цифр)")
    print(f"   2) итоговая оценка:  y = {y:.8g},  Δy = {dy:.3g}   ->  {r2}   ({n2} верных цифр)")


# а)  (0,62 + sqrt(16,9)) / lg 41,3
box_a = (num_i(0.62, -2) + isqrt(num_i(16.9, -1))) / ilg(num_i(41.3, -1))
task_161("а)  (0,62 + sqrt(16,9)) / lg 41,3", box_a,
         lambda p, q, r: (p + math.sqrt(q)) / math.log10(r),
         [0.62, 16.9, 41.3], [0.005, 0.05, 0.05])

# б)  (12,47 + sqrt(12,5^2 + 14,8^2)) / (sin^2 0,97 + cos^2 2,63)
chis = num_i(12.47, -2) + isqrt(ipow(num_i(12.5, -1), 2) + ipow(num_i(14.8, -1), 2))
znam = ipow(isin(num_i(0.97, -2)), 2) + ipow(icos(num_i(2.63, -2)), 2)
task_161("б)  (12,47 + sqrt(12,5^2 + 14,8^2)) / (sin^2 0,97 + cos^2 2,63)", chis / znam,
         lambda p, q, r, s, t: (p + math.sqrt(q * q + r * r)) / (math.sin(s) ** 2 + math.cos(t) ** 2),
         [12.47, 12.5, 14.8, 0.97, 2.63], [0.005, 0.05, 0.05, 0.005, 0.005])

# в)  ln(6,91 + 3,35^2) / sqrt(626,3)
box_v = iln(num_i(6.91, -2) + ipow(num_i(3.35, -2), 2)) / isqrt(num_i(626.3, -1))
task_161("в)  ln(6,91 + 3,35^2) / sqrt(626,3)", box_v,
         lambda p, q, r: math.log(p + q * q) / math.sqrt(r),
         [6.91, 3.35, 626.3], [0.005, 0.005, 0.05])

# г)  cbrt(26,88) / (e^3,94 - 8,04^2) + 6,19^1,34
e394 = iexp(num_i(3.94, -2))
kv804 = ipow(num_i(8.04, -2), 2)
razn = e394 - kv804
box_g = icbrt(num_i(26.88, -2)) / razn + ipow(num_i(6.19, -2), num_i(1.34, -2))
task_161("г)  cbrt(26,88) / (e^3,94 - 8,04^2) + 6,19^1,34", box_g,
         lambda p, q, r, s, t: (abs(p) ** (1 / 3)) / (math.exp(q) - r * r) + s ** t,
         [26.88, 3.94, 8.04, 6.19, 1.34], [0.005, 0.005, 0.005, 0.005, 0.005])

print("\n" + "-" * 78)
print("ОПАСНОЕ МЕСТО В ПУНКТЕ г): вычитание близких величин")
print(f"   e^3,94  = {e394}   отн. погрешность {e394.d() / abs(e394.mid()) * 100:.3f} %")
print(f"   8,04^2  = {kv804}   отн. погрешность {kv804.d() / abs(kv804.mid()) * 100:.3f} %")
print(f"   разность= {razn}   отн. погрешность {razn.d() / abs(razn.mid()) * 100:.2f} %")
print("   Относительная погрешность выросла примерно в 50 раз — это и съедает")
print("   верные цифры окончательного результата.")


УПРАЖНЕНИЕ 1.6.1

а)  (0,62 + sqrt(16,9)) / lg 41,3
   1) метод границ:     [2.919855643; 2.935475707]
      середина 2.9276657,  Δ = 0.00781   ->  2.9   (2 верных цифр)
   2) итоговая оценка:  y = 2.9276653,  Δy = 0.00781   ->  2.9   (2 верных цифр)

б)  (12,47 + sqrt(12,5^2 + 14,8^2)) / (sin^2 0,97 + cos^2 2,63)
   1) метод границ:     [21.91321276; 22.29201384]
      середина 22.102613,  Δ = 0.189   ->  22.0   (2 верных цифр)
   2) итоговая оценка:  y = 22.1011,  Δy = 0.189   ->  22.0   (2 верных цифр)

в)  ln(6,91 + 3,35^2) / sqrt(626,3)
   1) метод границ:     [0.1156983841; 0.1158773123]
      середина 0.11578785,  Δ = 8.95e-05   ->  0.116   (3 верных цифр)
   2) итоговая оценка:  y = 0.11578788,  Δy = 8.95e-05   ->  0.116   (3 верных цифр)

г)  cbrt(26,88) / (e^3,94 - 8,04^2) + 6,19^1,34
   1) метод границ:     [11.15526534; 11.40149414]
      середина 11.27838,  Δ = 0.123   ->  11.0   (2 верных цифр)
   2) итоговая оценка:  y = 11.277899,  Δy = 0.123   ->  11.0   (2 верных цифр

In [3]:
# ===== УПРАЖНЕНИЕ 1.6.2 =====
# a = 2,674 и b = 31,48 — все цифры верны в строгом смысле.
# Вычислить значения двумя способами:
#   1) с пооперационным учётом границ погрешностей;
#   2) с итоговой оценкой точности результата.

a_v, b_v = 2.674, 31.48
da, db = 0.0005, 0.005          # половина единицы последнего разряда

A_i, B_i = num_i(a_v, -3), num_i(b_v, -2)
A_e, B_e = E(a_v, da), E(b_v, db)

print("=" * 78)
print(f"УПРАЖНЕНИЕ 1.6.2    a = {a_v} (Δa = {da}),   b = {b_v} (Δb = {db})")
print("=" * 78)

results_162 = {}


def task_162(key, name, chain, f):
    y, dy = diff_err(f, (a_v, b_v), (da, db))
    m1, r1, n1 = verified(chain.v, chain.d)
    m2, r2, n2 = verified(y, dy)
    results_162[key] = (chain.d, dy)
    print(f"\n{name}")
    print(f"   1) пооперационно:    y = {chain.v:.8g},  Δy = {chain.d:.3g}   ->  {r1}   ({n1} верных цифр)")
    print(f"   2) итоговая оценка:  y = {y:.8g},  Δy = {dy:.3g}   ->  {r2}   ({n2} верных цифр)")
    print(f"      завышение пооперационного метода: в {chain.d / dy:.2f} раза")


task_162("а", "а)  a*b / sqrt(a + b^2)",
         (A_e * B_e) / esqrt(A_e + epow(B_e, 2)),
         lambda a, b: a * b / math.sqrt(a + b * b))

task_162("б", "б)  (a + sqrt(b)) / lg(a^2 + b^2)",
         (A_e + esqrt(B_e)) / elg(epow(A_e, 2) + epow(B_e, 2)),
         lambda a, b: (a + math.sqrt(b)) / math.log10(a * a + b * b))

task_162("в", "в)  (e^a - cbrt(b)) / ln(1 + a^2)",
         (eexp(A_e) - ecbrt(B_e)) / eln(1 + epow(A_e, 2)),
         lambda a, b: (math.exp(a) - b ** (1 / 3)) / math.log(1 + a * a))

task_162("г", "г)  lg[(cos^2 a + b) / (a^sqrt(b) + b^sqrt(a))]",
         elg((epow(ecos(A_e), 2) + B_e) / (epow(A_e, esqrt(B_e)) + epow(B_e, esqrt(A_e)))),
         lambda a, b: math.log10((math.cos(a) ** 2 + b) /
                                 (a ** math.sqrt(b) + b ** math.sqrt(a))))

print("\nВЫВОД: пооперационный учёт всегда даёт погрешность не меньше итоговой оценки —")
print("на каждом шаге погрешности складываются, даже когда реально частично гасят друг друга.")


УПРАЖНЕНИЕ 1.6.2    a = 2.674 (Δa = 0.0005),   b = 31.48 (Δb = 0.005)

а)  a*b / sqrt(a + b^2)
   1) пооперационно:    y = 2.6703996,  Δy = 0.00135   ->  2.67   (3 верных цифр)
   2) итоговая оценка:  y = 2.6703996,  Δy = 0.0005   ->  2.67   (4 верных цифр)
      завышение пооперационного метода: в 2.70 раза

б)  (a + sqrt(b)) / lg(a^2 + b^2)
   1) пооперационно:    y = 2.7623122,  Δy = 0.000443   ->  2.762   (4 верных цифр)
   2) итоговая оценка:  y = 2.7623122,  Δy = 0.000188   ->  2.762   (4 верных цифр)
      завышение пооперационного метода: в 2.35 раза

в)  (e^a - cbrt(b)) / ln(1 + a^2)
   1) пооперационно:    y = 5.4051733,  Δy = 0.00438   ->  5.41   (3 верных цифр)
   2) итоговая оценка:  y = 5.4051733,  Δy = 0.00269   ->  5.41   (3 верных цифр)
      завышение пооперационного метода: в 1.63 раза

г)  lg[(cos^2 a + b) / (a^sqrt(b) + b^sqrt(a))]
   1) пооперационно:    y = -1.216105,  Δy = 0.000557   ->  -1.22   (3 верных цифр)
   2) итоговая оценка:  y = -1.216105,  Δy = 0.0004

In [4]:
# ===== УПРАЖНЕНИЕ 1.6.3 =====
# Те же выражения вычислить по МЕТОДУ ГРАНИЦ: «вручную» и программным способом
# без пошаговой регистрации промежуточных границ; сравнить с результатами 1.6.2.

print("=" * 78)
print("УПРАЖНЕНИЕ 1.6.3    метод границ")
print("=" * 78)

funcs = {
    "а": ("a*b / sqrt(a + b^2)",
          lambda a, b: a * b / math.sqrt(a + b * b)),
    "б": ("(a + sqrt(b)) / lg(a^2 + b^2)",
          lambda a, b: (a + math.sqrt(b)) / math.log10(a * a + b * b)),
    "в": ("(e^a - cbrt(b)) / ln(1 + a^2)",
          lambda a, b: (math.exp(a) - b ** (1 / 3)) / math.log(1 + a * a)),
    "г": ("lg[(cos^2 a + b)/(a^sqrt(b) + b^sqrt(a))]",
          lambda a, b: math.log10((math.cos(a) ** 2 + b) /
                                  (a ** math.sqrt(b) + b ** math.sqrt(a)))),
}

# --- способ 1: пошаговая регистрация границ (интервальная арифметика) ---
step_boxes = {
    "а": (A_i * B_i) / isqrt(A_i + ipow(B_i, 2)),
    "б": (A_i + isqrt(B_i)) / ilg(ipow(A_i, 2) + ipow(B_i, 2)),
    "в": (iexp(A_i) - icbrt(B_i)) / iln(1 + ipow(A_i, 2)),
    "г": ilg((ipow(icos(A_i), 2) + B_i) /
             (ipow(A_i, isqrt(B_i)) + ipow(B_i, isqrt(A_i)))),
}


# --- способ 2: границы напрямую, без промежуточной регистрации ---
def direct_bounds(f, n=201):
    """Минимум и максимум выражения на прямоугольнике [a±Δa] x [b±Δb]."""
    vals = []
    for i in range(n):
        a = a_v - da + 2 * da * i / (n - 1)
        for j in range(n):
            b = b_v - db + 2 * db * j / (n - 1)
            vals.append(f(a, b))
    return I(min(vals), max(vals))


print(f"\n{'':4}{'способ':<34}{'нижняя':>15}{'верхняя':>15}{'Δ':>12}")
print("-" * 78)
for key, (label, f) in funcs.items():
    box_step = step_boxes[key]
    box_dir = direct_bounds(f)
    op_d, fin_d = results_162[key]
    print(f"\n{key}) {label}")
    print(f"{'':4}{'пошаговая регистрация границ':<34}{box_step.lo:>15.9g}{box_step.hi:>15.9g}{box_step.d():>12.3g}")
    print(f"{'':4}{'без регистрации (прямой минимакс)':<34}{box_dir.lo:>15.9g}{box_dir.hi:>15.9g}{box_dir.d():>12.3g}")
    print(f"{'':4}{'пооперационно (1.6.2, сп. 1)':<34}{'':>15}{'':>15}{op_d:>12.3g}")
    print(f"{'':4}{'итоговая оценка (1.6.2, сп. 2)':<34}{'':>15}{'':>15}{fin_d:>12.3g}")
    m, r, n = verified(box_dir.mid(), box_dir.d())
    print(f"{'':4}окончательно: {r}   ({n} верных цифр в строгом смысле)")
    print(f"{'':4}пошаговый метод шире истинного в {box_step.d() / box_dir.d():.2f} раза")

print("\n" + "=" * 78)
print("СРАВНЕНИЕ С 1.6.2")
print("=" * 78)
print("""
1. Прямой минимакс и итоговая оценка через дифференциал совпадают с точностью
   до третьего знака: обе дают ИСТИННУЮ ширину коридора значений.

2. Пошаговая регистрация границ и пооперационный учёт погрешностей дают одно
   и то же завышение. Причина одна: на каждой операции границы раздвигаются
   так, будто аргументы отклоняются независимо и всегда в худшую сторону.
   Реально a и b входят в выражение несколько раз и их отклонения частично
   компенсируются — пошаговый метод этого «не помнит».

3. Практический вывод: пошаговый метод безопасен (никогда не занижает
   погрешность), но при длинных цепочках операций теряет по одной верной
   цифре. Для окончательного ответа надёжнее итоговая оценка.
""")


УПРАЖНЕНИЕ 1.6.3    метод границ

    способ                                     нижняя        верхняя           Δ
------------------------------------------------------------------------------

а) a*b / sqrt(a + b^2)
    пошаговая регистрация границ           2.66905279     2.67174708     0.00135
    без регистрации (прямой минимакс)      2.66989985     2.67089944      0.0005
    пооперационно (1.6.2, сп. 1)                                         0.00135
    итоговая оценка (1.6.2, сп. 2)                                        0.0005
    окончательно: 2.67   (4 верных цифр в строгом смысле)
    пошаговый метод шире истинного в 2.70 раза

б) (a + sqrt(b)) / lg(a^2 + b^2)
    пошаговая регистрация границ           2.76186975     2.76275475    0.000443
    без регистрации (прямой минимакс)      2.76212417     2.76250028    0.000188
    пооперационно (1.6.2, сп. 1)                                        0.000443
    итоговая оценка (1.6.2, сп. 2)                                      0.00